# Differential Equations — Session 27
## Section 6.2: Power-Series Solutions About Ordinary Points

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Classify ordinary and singular points; state the ordinary-point existence theorem; estimate a guaranteed radius from the nearest singularity; derive recurrences; construct two independent series; apply initial data; and compare truncated series with numerical solutions.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Ordinary and singular points |
| 18–35 min | Nearest-singularity theorem |
| 35–65 min | Airy-type recurrence |
| 65–80 min | Two independent series and IVPs |
| 80–90 min | Numerical comparison |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import jv, yv, iv, kv, eval_legendre, jn_zeros
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)
def polynomial_value(coefficients, x):
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for n, c in enumerate(coefficients):
        total += c*x**n
    return total
print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 6.2-A — Ordinary point

For
$$
a_2(x)y''+a_1(x)y'+a_0(x)y=0,
$$
write
$$
y''+P(x)y'+Q(x)y=0.
$$
A point $x_0$ is ordinary when $P$ and $Q$ are analytic there; otherwise it is singular.

### Theorem 6.2-B — Existence of ordinary-point series solutions

If $x_0$ is ordinary, then there exist two linearly independent solutions
$$
y_1=\sum_{n=0}^{\infty}a_n(x-x_0)^n,
\qquad
y_2=\sum_{n=0}^{\infty}b_n(x-x_0)^n.
$$
Each converges at least to the nearest singular point in the complex plane.

### Corollary 6.2-C — Initial coefficients

For a series centered at $x_0$,
$$
y(x_0)=c_0,\qquad y'(x_0)=c_1.
$$

### Method 6.2-D — Undetermined series coefficients

Assume a series, differentiate, substitute, align powers, equate coefficients, derive the recurrence, and group the two free coefficient chains.

### Classroom Checkpoint — Guaranteed Radius

For a power series centered at $x_0$, what geometric quantity determines the guaranteed radius of convergence of an ordinary-point solution?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Locate singular points

For
$$
(1-x^2)y''+xy'+4y=0,
$$
the singular points are $x=\pm1$.

In [ ]:
x = np.linspace(-3, 3, 500)
plt.plot(x, 1-x**2)
plt.axhline(0, linestyle="--")
plt.scatter([-1, 1], [0, 0], s=90, label="singular points")
plt.legend()
plt.title("Zeros of the leading coefficient")
plt.show()

## 2. Distance to complex singularities

If singularities are $1\pm2i$, a series centered at $x_0$ is guaranteed to converge at least to the closer one.

In [ ]:
def singularity_distance(x0=0.0):
    singularities = np.array([1+2j, 1-2j])
    R = np.min(np.abs(singularities-x0))
    plt.scatter(singularities.real, singularities.imag, s=100, label="singularities")
    plt.scatter([x0], [0], s=100, label="center")
    plt.gca().add_patch(plt.Circle((x0, 0), R, fill=False, linestyle="--"))
    plt.axhline(0); plt.axvline(0)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlim(-4, 5); plt.ylim(-4, 4)
    plt.title(fr"Guaranteed radius $R={R:.3f}$")
    plt.legend(); plt.show()
if WIDGETS_AVAILABLE:
    interact(singularity_distance, x0=FloatSlider(min=-3, max=4, step=0.1, value=0))
else:
    singularity_distance()

## 3. Airy-type equation

For
$$
y''-xy=0,
$$
substitution of $y=\sum c_nx^n$ gives
$$
c_2=0,
\qquad
c_{n+2}=\frac{c_{n-1}}{(n+2)(n+1)},\quad n\ge1.
$$
The recurrence advances by three indices.

In [ ]:
def airy_coefficients(c0=1.0, c1=0.0, N=18):
    c = np.zeros(N+1)
    c[0] = c0
    if N >= 1: c[1] = c1
    if N >= 2: c[2] = 0
    for n in range(1, N-1):
        c[n+2] = c[n-1]/((n+2)*(n+1))
    return c

for initial in [(1, 0), (0, 1)]:
    print(initial, airy_coefficients(*initial, 15))

In [ ]:
x = np.linspace(-4, 4, 800)
for N in [4, 8, 14, 20]:
    plt.plot(x, polynomial_value(airy_coefficients(1, 0, N), x), label=f"N={N}")
plt.ylim(-20, 80)
plt.legend()
plt.title("Truncated Airy-type series")
plt.show()

## 4. Numerical comparison

In [ ]:
def rhs(x, z):
    return [z[1], x*z[0]]

grid = np.linspace(-3.5, 3.5, 700)
sol = solve_ivp(rhs, (-3.5, 3.5), [1, 0], t_eval=grid, rtol=1e-10, atol=1e-12)
series = polynomial_value(airy_coefficients(1, 0, 24), grid)
plt.plot(grid, sol.y[0], label="numerical")
plt.plot(grid, series, linestyle="--", label="degree 24")
plt.ylim(-15, 50)
plt.legend()
plt.show()
print("Maximum grid error:", np.max(np.abs(sol.y[0]-series)))

In [ ]:
def airy_truncation(N=12, c0=1.0, c1=0.0):
    grid = np.linspace(-3, 3, 600)
    def rhs(t, z): return [z[1], t*z[0]]
    sol = solve_ivp(rhs, (-3, 3), [c0, c1], t_eval=grid, rtol=1e-10, atol=1e-12)
    approx = polynomial_value(airy_coefficients(c0, c1, N), grid)
    plt.plot(grid, sol.y[0], label="numerical")
    plt.plot(grid, approx, linestyle="--", label=f"degree {N}")
    plt.ylim(-12, 40)
    plt.legend(); plt.show()
if WIDGETS_AVAILABLE:
    interact(airy_truncation,
             N=IntSlider(min=3, max=30, value=12),
             c0=FloatSlider(min=-2, max=2, step=0.25, value=1),
             c1=FloatSlider(min=-2, max=2, step=0.25, value=0))
else:
    airy_truncation()

## 5. Nonpolynomial coefficients

For $y''+(\cos x)y=0$, the coefficient is analytic everywhere. Multiplication of the cosine series and the unknown series produces a convolution recurrence.

In [ ]:
x = sp.symbols("x")
N = 10
c = sp.symbols(f"c0:{N+1}")
y = sum(c[n]*x**n for n in range(N+1))
expr = sp.series(sp.diff(y, x, 2)+sp.cos(x)*y, x, 0, 7).removeO().expand()
poly = sp.Poly(expr, x)
for power in range(5):
    display(sp.Eq(poly.coeff_monomial(x**power), 0))

## Exit check

For
$$
(x^2+4)y''+xy'+y=0,
$$
the singularities are $\pm2i$, so a series centered at zero has guaranteed radius $R=2$.

## Classroom Checkpoint — Closing Reflection

Before continuing, try to state the central method or theorem of this lesson, including its assumptions and one situation in which it is useful.

> Discuss first; run the next cell for an instructor summary.